# WASH Sub-Saharan Africa Analysis — Data Cleaning & Preparation
**Project:** Community Water and Sanitation Access Survey Analysis (WASH) in Sub-Saharan Africa  
**Data Source:** WHO/UNICEF Joint Monitoring Programme (JMP) 2025 Global Dataset  
**Tool:** Python (pandas)  
**Author:** Clement Etim  
**Date:** May 2026

---

## About This Notebook
This notebook covers Phase 1 of the analysis: loading, filtering, cleaning,
and exporting the JMP global WASH dataset for Africa-focused analysis.

The JMP dataset tracks access to drinking water, sanitation, and hygiene
services across 200+ countries from 2000 to 2024. Our goal is to extract
sub-Saharan African countries only and prepare a clean dataset for exploratory analysis
and Power BI visualisation.

## Step 1 — Install Required Libraries
We install `xlsxwriter` to enable Excel export with multiple sheets.

In [1]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.2 MB/s eta 0:00:00


## Step 2 — Import Libraries
We import `pandas`, the core Python library for data manipulation.
It allows us to load, filter, reshape, and export tabular data efficiently.

In [2]:
import pandas as pd


## Step 3 — Load the Raw Dataset
The JMP Excel file contains 9 sheets. We load only the three
sheets relevant to our analysis:
- `wat` — Drinking water access data
- `san` — Sanitation facilities data
- `hyg` — Hygiene (handwashing) data

Each sheet contains one row per country per year, covering 2000–2024.

In [3]:
file_path = "/content/JMP_2025_WLD.xlsx"

wat = pd.read_excel(file_path, sheet_name="wat")
san = pd.read_excel(file_path, sheet_name="san")
hyg = pd.read_excel(file_path, sheet_name="hyg")

print("Water shape:", wat.shape)
print("Sanitation shape:", san.shape)
print("Hygiene shape:", hyg.shape)

Water shape: (5661, 59)
Sanitation shape: (5651, 65)
Hygiene shape: (3701, 38)


## Step 4 — Explore Region Classifications
Before filtering for Africa, we inspect the region columns available in the
dataset. The JMP uses multiple regional classification systems (SDG, UNICEF,
WHO). We need to identify which one gives us the cleanest Africa-only filter.

In [4]:
print("SDG Regions:")
print(wat["region_sdg"].unique())

print("\nUNICEF Regions:")
print(wat["region_unicef_programme"].unique())

SDG Regions:
['Central and Southern Asia' 'Europe and Northern America'
 'Northern Africa and Western Asia' 'Oceania' 'Sub-Saharan Africa'
 'Latin America and the Caribbean' 'Australia and New Zealand'
 'Eastern and South-Eastern Asia']

UNICEF Regions:
['South Asia' 'Europe and Central Asia' 'Middle East and North Africa' nan
 'Eastern and Southern Africa' 'Latin America and the Caribbean'
 'West and Central Africa' 'East Asia and Pacific']


## Step 5 — Filter for sub-Saharan African Countries
We filter the dataset to retain only sub-Saharan African countries using the UNICEF
Programme Region column. We use this instead of the SDG Region column because
the SDG classification groups Northern African countries together with Western
Asian (Middle Eastern) countries, which would introduce non-African data
into our analysis.

The two UNICEF regions that cover Africa are:
- Eastern and Southern Africa
- West and Central Africa

This filter is applied to all three sheets: water, sanitation, and hygiene.

In [5]:
africa_regions = [
    "Eastern and Southern Africa",
    "West and Central Africa"
]

wat_africa = wat[wat["region_unicef_programme"].isin(africa_regions)].copy()
san_africa = san[san["region_unicef_programme"].isin(africa_regions)].copy()
hyg_africa = hyg[hyg["region_unicef_programme"].isin(africa_regions)].copy()

print("Water — Africa rows:", len(wat_africa), "| Countries:", wat_africa["name"].nunique())
print("Sanitation — Africa rows:", len(san_africa), "| Countries:", san_africa["name"].nunique())
print("Hygiene — Africa rows:", len(hyg_africa), "| Countries:", hyg_africa["name"].nunique())

Water — Africa rows: 1096 | Countries: 45
Sanitation — Africa rows: 1094 | Countries: 45
Hygiene — Africa rows: 770 | Countries: 43


## Step 6 — See Filtered Dataset
Before we move head, we print a summary of our filtered dataset
to confirm.

In [6]:
print("=" * 50)
print("ANALYSIS SCOPE: Sub-Saharan Africa")
print("=" * 50)

print(f"\nYear coverage: {wat_africa['year'].min()} - {wat_africa['year'].max()}")

print(f"\nCountries included: {wat_africa['name'].nunique()}")
print(sorted(wat_africa['name'].unique()))

print(f"\nRows per sheet:")
print(f"  Water:       {len(wat_africa)}")
print(f"  Sanitation:  {len(san_africa)}")
print(f"  Hygiene:     {len(hyg_africa)}")

ANALYSIS SCOPE: Sub-Saharan Africa

Year coverage: 2000 - 2024

Countries included: 45
['Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cameroon', 'Central African Republic', 'Chad', 'Comoros', 'Congo', "Côte d'Ivoire", 'Democratic Republic of the Congo', 'Equatorial Guinea', 'Eritrea', 'Eswatini', 'Ethiopia', 'Gabon', 'Gambia', 'Ghana', 'Guinea', 'Guinea-Bissau', 'Kenya', 'Lesotho', 'Liberia', 'Madagascar', 'Malawi', 'Mali', 'Mauritania', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda', 'Sao Tome and Principe', 'Senegal', 'Sierra Leone', 'Somalia', 'South Africa', 'South Sudan', 'Togo', 'Uganda', 'United Republic of Tanzania', 'Zambia', 'Zimbabwe']

Rows per sheet:
  Water:       1096
  Sanitation:  1094
  Hygiene:     770


## Step 7 — Renaming Columns
The raw JMP dataset contain many columns we don't need for this
analysis, and the column names are short codes that are difficult
to read (e.g. "wat_basal_r" instead of "Water_Basic_Rural").

In this step we:
1. Select only the columns that are relevant to our analysis
2. Rename them into clear and readable labels

We do this separately for water, sanitation, and hygiene.


In [7]:
wat_cols = {
    "name": "Country",
    "iso3": "ISO3",
    "year": "Year",
    "pop_t": "Population_thousands",
    "prop_u": "Pct_Urban",
    "region_unicef_programme": "UNICEF_Region",
    "region_sdg": "SDG_Region",
    "region_income": "Income_Group",
    "wat_basal_r": "Water_Basic_Rural",
    "wat_sm_r": "Water_SafelyManaged_Rural",
    "wat_unimp_r": "Water_Unimproved_Rural",
    "wat_ns_r": "Water_SurfaceWater_Rural",
    "wat_basal_u": "Water_Basic_Urban",
    "wat_sm_u": "Water_SafelyManaged_Urban",
    "wat_unimp_u": "Water_Unimproved_Urban",
    "wat_ns_u": "Water_SurfaceWater_Urban",
    "wat_basal_t": "Water_Basic_Total",
    "wat_sm_t": "Water_SafelyManaged_Total",
    "wat_unimp_t": "Water_Unimproved_Total",
    "wat_ns_t": "Water_SurfaceWater_Total",
}

wat_clean = wat_africa[list(wat_cols.keys())].rename(columns=wat_cols)
print("Shape:", wat_clean.shape)
print("Columns:", list(wat_clean.columns))

Shape: (1096, 20)
Columns: ['Country', 'ISO3', 'Year', 'Population_thousands', 'Pct_Urban', 'UNICEF_Region', 'SDG_Region', 'Income_Group', 'Water_Basic_Rural', 'Water_SafelyManaged_Rural', 'Water_Unimproved_Rural', 'Water_SurfaceWater_Rural', 'Water_Basic_Urban', 'Water_SafelyManaged_Urban', 'Water_Unimproved_Urban', 'Water_SurfaceWater_Urban', 'Water_Basic_Total', 'Water_SafelyManaged_Total', 'Water_Unimproved_Total', 'Water_SurfaceWater_Total']


In [8]:
san_cols = {
    "name": "Country",
    "iso3": "ISO3",
    "year": "Year",
    "pop_t": "Population_thousands",
    "prop_u": "Pct_Urban",
    "region_unicef_programme": "UNICEF_Region",
    "region_sdg": "SDG_Region",
    "region_income": "Income_Group",
    "san_basal_r": "San_Basic_Rural",
    "san_sm_r": "San_SafelyManaged_Rural",
    "san_ns_r": "San_OpenDefecation_Rural",
    "san_lim_r": "San_Limited_Rural",
    "san_basal_u": "San_Basic_Urban",
    "san_sm_u": "San_SafelyManaged_Urban",
    "san_ns_u": "San_OpenDefecation_Urban",
    "san_lim_u": "San_Limited_Urban",
    "san_basal_t": "San_Basic_Total",
    "san_sm_t": "San_SafelyManaged_Total",
    "san_ns_t": "San_OpenDefecation_Total",
    "san_lim_t": "San_Limited_Total",
}

san_clean = san_africa[list(san_cols.keys())].rename(columns=san_cols)
print("Shape:", san_clean.shape)
print("Columns:", list(san_clean.columns))

Shape: (1094, 20)
Columns: ['Country', 'ISO3', 'Year', 'Population_thousands', 'Pct_Urban', 'UNICEF_Region', 'SDG_Region', 'Income_Group', 'San_Basic_Rural', 'San_SafelyManaged_Rural', 'San_OpenDefecation_Rural', 'San_Limited_Rural', 'San_Basic_Urban', 'San_SafelyManaged_Urban', 'San_OpenDefecation_Urban', 'San_Limited_Urban', 'San_Basic_Total', 'San_SafelyManaged_Total', 'San_OpenDefecation_Total', 'San_Limited_Total']


In [9]:
hyg_cols = {
    "name": "Country",
    "iso3": "ISO3",
    "year": "Year",
    "pop_t": "Population_thousands",
    "prop_u": "Pct_Urban",
    "region_unicef_programme": "UNICEF_Region",
    "region_sdg": "SDG_Region",
    "region_income": "Income_Group",
    "hyg_bas_r": "Hyg_Basic_Rural",
    "hyg_lim_r": "Hyg_Limited_Rural",
    "hyg_ns_r": "Hyg_NoFacility_Rural",
    "hyg_bas_u": "Hyg_Basic_Urban",
    "hyg_lim_u": "Hyg_Limited_Urban",
    "hyg_ns_u": "Hyg_NoFacility_Urban",
    "hyg_bas_t": "Hyg_Basic_Total",
    "hyg_lim_t": "Hyg_Limited_Total",
    "hyg_ns_t": "Hyg_NoFacility_Total",
}

hyg_clean = hyg_africa[list(hyg_cols.keys())].rename(columns=hyg_cols)
print("Shape:", hyg_clean.shape)
print("Columns:", list(hyg_clean.columns))

Shape: (770, 17)
Columns: ['Country', 'ISO3', 'Year', 'Population_thousands', 'Pct_Urban', 'UNICEF_Region', 'SDG_Region', 'Income_Group', 'Hyg_Basic_Rural', 'Hyg_Limited_Rural', 'Hyg_NoFacility_Rural', 'Hyg_Basic_Urban', 'Hyg_Limited_Urban', 'Hyg_NoFacility_Urban', 'Hyg_Basic_Total', 'Hyg_Limited_Total', 'Hyg_NoFacility_Total']


## Step 8 — Check for Missing Values
Missing values in the JMP dataset typically occur because:
- A country did not report data for a particular year
- A specific indicator was not measured in that country
- Data was suppressed due to quality concerns

In [10]:
print("=" * 50)
print("MISSING VALUES SUMMARY")
print("=" * 50)

for label, df in [("Water", wat_clean), ("Sanitation", san_clean), ("Hygiene", hyg_clean)]:
    print(f"\n--- {label} ---")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print("No missing values.")
    else:
        missing_pct = (missing / len(df) * 100).round(1)
        summary = pd.DataFrame({"Missing_Count": missing, "Missing_Pct": missing_pct})
        print(summary)

MISSING VALUES SUMMARY

--- Water ---
                           Missing_Count  Missing_Pct
Water_SafelyManaged_Rural            499         45.5
Water_SurfaceWater_Rural               1          0.1
Water_SafelyManaged_Urban            424         38.7
Water_SafelyManaged_Total            474         43.2
Water_SurfaceWater_Total               1          0.1

--- Sanitation ---
                         Missing_Count  Missing_Pct
San_SafelyManaged_Rural            346         31.6
San_SafelyManaged_Urban            369         33.7
San_SafelyManaged_Total            371         33.9

--- Hygiene ---
                      Missing_Count  Missing_Pct
Pct_Urban                         1          0.1
Hyg_Basic_Rural                  62          8.1
Hyg_Limited_Rural                88         11.4
Hyg_NoFacility_Rural             51          6.6
Hyg_Basic_Urban                  74          9.6
Hyg_Limited_Urban                92         11.9
Hyg_NoFacility_Urban             64          8.3
H

## Step 9 — Handling Missing Values
All missing values are left as NaN without imputation. This is standard
practice in international development data analysis.

## Step 10 — Making Two Dataset Versions
We create two versions of each cleaned dataset for different purposes:

1. Time Series (2000–2024): the full dataset for trend analysis
   and tracking progress over time.
2. Latest Year Snapshot: one row per country using the most recent
   available data, which is going to be used for cross-country comparisons and the
   Power BI dashboard.

In [11]:
# Time series — full 2000-2024
wat_trend = wat_clean.copy()
san_trend = san_clean.copy()
hyg_trend = hyg_clean.copy()

# Latest year — one row per country
wat_latest = wat_clean.sort_values("Year").groupby("Country").last().reset_index()
san_latest = san_clean.sort_values("Year").groupby("Country").last().reset_index()
hyg_latest = hyg_clean.sort_values("Year").groupby("Country").last().reset_index()

print("Time Series:")
print(f"  Water:       {wat_trend.shape}")
print(f"  Sanitation:  {san_trend.shape}")
print(f"  Hygiene:     {hyg_trend.shape}")

print("\nLatest Year Snapshot:")
print(f"  Water:       {wat_latest.shape}")
print(f"  Sanitation:  {san_latest.shape}")
print(f"  Hygiene:     {hyg_latest.shape}")

Time Series:
  Water:       (1096, 20)
  Sanitation:  (1094, 20)
  Hygiene:     (770, 17)

Latest Year Snapshot:
  Water:       (45, 20)
  Sanitation:  (45, 20)
  Hygiene:     (43, 17)


## Step 11 — Export Cleaned Datasets
We export the cleaned datasets in two formats:

1. Excel File (.xlsx): a single workbook with six tabs. One time series
   and one latest year snapshot for each of the three themes.

2. CSV files: individual files for each dataset, to be used for Power BI dashboard.

In [12]:
# Export to Excel
output_path = "/content/africa_wash_clean.xlsx"

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    wat_trend.to_excel(writer, sheet_name="Water_TimeSeries", index=False)
    san_trend.to_excel(writer, sheet_name="San_TimeSeries", index=False)
    hyg_trend.to_excel(writer, sheet_name="Hyg_TimeSeries", index=False)
    wat_latest.to_excel(writer, sheet_name="Water_Latest", index=False)
    san_latest.to_excel(writer, sheet_name="San_Latest", index=False)
    hyg_latest.to_excel(writer, sheet_name="Hyg_Latest", index=False)

print("Excel export complete:", output_path)

# Export to CSV
wat_trend.to_csv("/content/water_timeseries.csv", index=False)
san_trend.to_csv("/content/sanitation_timeseries.csv", index=False)
hyg_trend.to_csv("/content/hygiene_timeseries.csv", index=False)
wat_latest.to_csv("/content/water_latest.csv", index=False)
san_latest.to_csv("/content/sanitation_latest.csv", index=False)
hyg_latest.to_csv("/content/hygiene_latest.csv", index=False)

print("CSV exports complete.")
print("\nFiles ready for download in the Files panel.")

Excel export complete: /content/africa_wash_clean.xlsx
CSV exports complete.

Files ready for download in the Files panel.
